# MedNorm Phase 2 Validation Ablation

Status: READY_FOR_COLAB_SMOKE. Intended environment: Colab CPU or GPU after E5, proposal, and learned-L4 artifacts exist on Drive. This notebook validates artifacts, runs governed-validation-only ablations, freezes a candidate Phase-2 profile, and prepares the future internal_test gate. It writes reports under `/content/drive/MyDrive/MedNorm-VI/artifacts/phase2_validation_ablation_v1`. It does not run internal_test without `I_AUTHORIZE_ONE_SHOT_INTERNAL_TEST_PHASE2`, never runs organizer inference, and never writes `output.zip`. Validation ablation authorization is `I_AUTHORIZE_PHASE2_VALIDATION_ABLATION`.

**E4 arm removed (Audit 0056a).** E4 PhoBERT-W2NER is `RETIRED_FROM_ACTIVE_ARCHITECTURE` (Audits 0043-0048, 0051). Its arm was still executable here: this notebook imported `validate_e4_artifact`, a symbol that **no longer exists** in `mednorm_vi.training.phase2.artifacts`, so cell 3 would have raised `ImportError` on the first run; and it set `enable_e4_phobert_w2ner`, a flag `PipelineConfig.load` now **refuses** outright. The architecture-valid arm set is E3, E5, E6 and learned L4 v2.

In [ ]:
from pathlib import Path
import json

DRIVE_ROOT = Path("/content/drive/MyDrive/MedNorm-VI")
REPO_DIR = Path("/content/MedNorm-VI")
VALIDATION_REPORT_DIR = DRIVE_ROOT / "artifacts" / "phase2_validation_ablation_v1"
VALIDATION_REPORT_DIR.mkdir(parents=True, exist_ok=True)
# E4_FULL_ARTIFACT was removed in Audit 0056a: E4 is retired and no E4 artifact is
# produced, retained or validated anywhere in the active architecture.
E5_FULL_ARTIFACT = DRIVE_ROOT / "artifacts" / "e5_xlmr_mrc_full_v1"
L4_FULL_ARTIFACT = DRIVE_ROOT / "artifacts" / "l4_learned_resolver_v2_full_v1"
RUN_VALIDATION_ABLATION = False
CONFIRM_VALIDATION_ABLATION = ""
CONFIRM_INTERNAL_TEST = ""
if RUN_VALIDATION_ABLATION and CONFIRM_VALIDATION_ABLATION != "I_AUTHORIZE_PHASE2_VALIDATION_ABLATION":
    raise SystemExit("validation ablation requires explicit operator authorization")


In [ ]:
from mednorm_vi.training.phase2.artifacts import validate_e5_artifact, validate_l4_artifact

# `validate_e4_artifact` was imported here until Audit 0056a. It does not exist in
# `mednorm_vi.training.phase2.artifacts` — E4 was retired in Audit 0048 and its
# validator went with it — so this cell raised ImportError before doing anything.
artifact_reports = {
    "e5": validate_e5_artifact(E5_FULL_ARTIFACT),
    "l4": validate_l4_artifact(L4_FULL_ARTIFACT),
}
for name, report in artifact_reports.items():
    print(name, json.dumps(report.as_dict(), indent=2, sort_keys=True))


In [ ]:
from mednorm_vi.evaluation.l3_l4_ablation_v2 import plan_phase2_ablation
from mednorm_vi.inference.config import DEFAULT_FEATURE_FLAGS

feature_flags = dict(DEFAULT_FEATURE_FLAGS)
# The architecture-valid arm components. `e4_phobert_w2ner` was listed here until
# Audit 0056a; E4 is retired and produces no checkpoint.
checkpoint_paths = {
    "e3_vihealthbert": "checkpoint/s1_mention_full_training_v1/best.pt",
    "e5_xlmr_mrc": str(E5_FULL_ARTIFACT / "checkpoints" / "best.pt") if artifact_reports["e5"].ok else "",
    "l4_learned_v2": str(L4_FULL_ARTIFACT / "checkpoints" / "best.pt") if artifact_reports["l4"].ok else "",
}
# `enable_e4_phobert_w2ner` was in this loop until Audit 0056a. `PipelineConfig.load`
# now REFUSES any profile declaring it, so setting it here could only produce a
# config that cannot be loaded.
for flag in ("enable_e5_xlmr_mrc", "enable_l4_learned_v2"):
    feature_flags[flag] = False
arm_statuses = plan_phase2_ablation(feature_flags, checkpoint_paths)
print(json.dumps([status.as_dict() for status in arm_statuses], indent=2, sort_keys=True))


In [ ]:
from mednorm_vi.training.phase2.validation_ablation import run_validation_ablation, write_validation_ablation_report

validation_examples = []
predictions_by_arm = {}
config_hashes_by_arm = {}
checkpoint_hashes_by_arm = {}
if RUN_VALIDATION_ABLATION:
    report = run_validation_ablation(
        validation_examples,
        predictions_by_arm,
        arm_statuses=arm_statuses,
        config_hashes_by_arm=config_hashes_by_arm,
        checkpoint_hashes_by_arm=checkpoint_hashes_by_arm,
    )
    validation_ablation_hash = write_validation_ablation_report(VALIDATION_REPORT_DIR / "validation_ablation_report.json", report)
else:
    validation_ablation_hash = ""
print(json.dumps({"validation_ablation_hash": validation_ablation_hash, "internal_test_accessed": False}, indent=2, sort_keys=True))


In [ ]:
from mednorm_vi.training.phase2.internal_test_gate import INTERNAL_TEST_AUTHORIZATION, evaluate_internal_test_freeze_gate

profile_manifest = {
    "feature_flags": feature_flags,
    "thresholds": {"learned_l4_keep_threshold": 0.5, "learned_l4_wrong_type_risk_threshold": 0.35},
    "config_hashes": config_hashes_by_arm,
    "checkpoint_hashes": checkpoint_hashes_by_arm,
    "validation_ablation_complete": bool(validation_ablation_hash),
    "validation_ablation_hash": validation_ablation_hash,
    "model_revisions": {},
    "internal_test_accessed": False,
}
(VALIDATION_REPORT_DIR / "phase2_candidate_profile_manifest.json").write_text(json.dumps(profile_manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
gate = evaluate_internal_test_freeze_gate(
    artifact_reports=tuple(artifact_reports.values()),
    frozen_feature_flags=profile_manifest["feature_flags"],
    frozen_thresholds=profile_manifest["thresholds"],
    config_hashes=profile_manifest["config_hashes"],
    checkpoint_hashes=profile_manifest["checkpoint_hashes"],
    validation_ablation_complete=profile_manifest["validation_ablation_complete"],
    validation_ablation_hash=profile_manifest["validation_ablation_hash"],
    model_revisions=profile_manifest["model_revisions"],
    authorization=CONFIRM_INTERNAL_TEST,
)
print(json.dumps(gate.as_dict(), indent=2, sort_keys=True))
assert CONFIRM_INTERNAL_TEST in ("", INTERNAL_TEST_AUTHORIZATION)
